# 画像から 3D の試作(ステップ 1・第1回)

素体の水着の絵(3人)から、Hunyuan3D-2(1枚・多視点)と TRELLIS.2(1枚)で 3D を作って比べる。

1. ランタイムを **GPU(L4 か A100)** にする。
2. 上から順に実行する。セル 1 で `m3d_inputs.zip` を選んでアップロードする。
3. **Hunyuan3D-2 のあと、TRELLIS.2 の前に「ランタイムを再起動」する**(ライブラリがぶつかるため)。再起動後はセル 1 からやり直さず、
   セル「TRELLIS.2: 準備」から実行する(入力と結果は /content に残る)。
4. 最後のセルで `m3d_results.zip` をダウンロードして、アプリの作業者に渡す。

結果は `/content/results/<キャラ>/<方法>.glb` と、時間・VRAM の記録 `log.json`。失敗した方法は log にエラーを残して次へ進む。

In [ ]:
# セル 1: GPU の確認と入力のアップロード
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, zipfile
from google.colab import files
os.chdir('/content')
if not os.path.isdir('/content/inputs'):
    up = files.upload()  # m3d_inputs.zip を選ぶ
    zipfile.ZipFile(next(iter(up))).extractall('/content')
print(sorted(os.listdir('/content/inputs')))

In [ ]:
# セル 2: 記録の道具(時間・VRAM のピーク・エラー)
import os, json, time, traceback, torch, gc
os.makedirs('/content/results', exist_ok=True)
LOG = '/content/results/log.json'

def log(char, method, **kw):
    data = json.load(open(LOG)) if os.path.exists(LOG) else []
    data.append({'char': char, 'method': method, 'gpu': torch.cuda.get_device_name(0), **kw})
    json.dump(data, open(LOG, 'w'), ensure_ascii=False, indent=1)
    print(char, method, kw)

def run(char, method, fn):
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    try:
        fn()
        log(char, method, ok=True, seconds=round(time.time() - t, 1),
            vram_gb=round(torch.cuda.max_memory_allocated() / 2**30, 1))
    except Exception as exc:
        log(char, method, ok=False, error=f'{type(exc).__name__}: {exc}', trace=traceback.format_exc()[-1500:])
    gc.collect(); torch.cuda.empty_cache()

CHARS = sorted(os.listdir('/content/inputs'))

## Hunyuan3D-2(1枚の A ポーズ・多視点の正面/横/後ろ)

In [ ]:
# Hunyuan3D-2: 準備(10分ほど。テクスチャの部品をビルドする)
%cd /content
!git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q -e .
# 新しい transformers(5)では、画像の部品(DINOv2)の重みの名前が変わり、配布の重みを読めない
!pip install -q "transformers>=4.46,<5"
# transformers を戻すと huggingface-hub も古くなり、新しい diffusers(テクスチャを塗るのに使う)と合わなくなる
!pip install -q "diffusers>=0.30,<0.36"
# 「pip's dependency resolver」の ERROR は、Colab に入っている別のライブラリ(cudf・gradio など)の警告で、ここでは使わない
%cd hy3dgen/texgen/custom_rasterizer
!python3 setup.py install > /content/build_rasterizer.log 2>&1; tail -2 /content/build_rasterizer.log
%cd ../differentiable_renderer
!python3 setup.py install > /content/build_renderer.log 2>&1; tail -2 /content/build_renderer.log
%cd /content/Hunyuan3D-2
# pip で numpy などが入れ替わったので、読み込み直すためにセッションを再起動する(「クラッシュ」と出るが正常)。
# 再起動のあとは、このセルをやり直さず、次のセルから実行する
import os
os.kill(os.getpid(), 9)

In [ ]:
# Hunyuan3D-2: 1枚(A ポーズ)と多視点(正面・左・後ろ)。形を作ってからテクスチャを塗る
# (再起動のあとでも動くよう、記録の道具もここで読み込む)
import os, sys, json, time, traceback, gc, torch
LOG = '/content/results/log.json'
os.makedirs('/content/results', exist_ok=True)
def log(char, method, **kw):
    data = json.load(open(LOG)) if os.path.exists(LOG) else []
    data.append({'char': char, 'method': method, 'gpu': torch.cuda.get_device_name(0), **kw})
    json.dump(data, open(LOG, 'w'), ensure_ascii=False, indent=1)
    print(char, method, kw)
def run(char, method, fn):
    torch.cuda.reset_peak_memory_stats(); t = time.time()
    try:
        fn(); log(char, method, ok=True, seconds=round(time.time() - t, 1),
                  vram_gb=round(torch.cuda.max_memory_allocated() / 2**30, 1))
    except Exception as exc:
        log(char, method, ok=False, error=f'{type(exc).__name__}: {exc}', trace=traceback.format_exc()[-1500:])
    gc.collect(); torch.cuda.empty_cache()
CHARS = sorted(os.listdir('/content/inputs'))
os.chdir('/content/Hunyuan3D-2')
sys.path.insert(0, '/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline

rembg = BackgroundRemover()
def cut(path):
    # 素体の絵は灰色の背景なので、背景を消して RGBA にする
    return rembg(Image.open(path).convert('RGB'))

single = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
for char in CHARS:
    src = f'/content/inputs/{char}'
    out = f'/content/results/{char}'; os.makedirs(out, exist_ok=True)
    def one():
        image = cut(f'{src}/a_pose.png'); image.save(f'{out}/hy2_single_input.png')
        mesh = single(image=image, generator=torch.manual_seed(12345))[0]
        mesh.export(f'{out}/hy2_single_shape.glb')
        paint(mesh, image=image).export(f'{out}/hy2_single.glb')
    run(char, 'hy2_single', one)
del single; gc.collect(); torch.cuda.empty_cache()

mv = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mv', subfolder='hunyuan3d-dit-v2-mv', variant='fp16')
for char in CHARS:
    src = f'/content/inputs/{char}'
    out = f'/content/results/{char}'
    def many():
        views = {k: cut(f'{src}/{k}.png') for k in ('front', 'left', 'back')}
        mesh = mv(image=views, num_inference_steps=50, octree_resolution=380, num_chunks=20000,
                  generator=torch.manual_seed(12345), output_type='trimesh')[0]
        mesh.export(f'{out}/hy2_mv_shape.glb')
        paint(mesh, image=views['front']).export(f'{out}/hy2_mv.glb')
    run(char, 'hy2_mv', many)
del mv, paint; gc.collect(); torch.cuda.empty_cache()
print(open(LOG).read())

## TRELLIS.2(1枚の A ポーズ)

**ここで「ランタイム > セッションを再起動」してから**、下のセルを順に実行する。24GB 以上が要る(L4 はぎりぎり、A100 なら余裕)。
準備(setup.sh)が失敗したら、エラーの最後の数行を控えてこの節は飛ばしてよい。

In [ ]:
# TRELLIS.2: 準備(15〜30分。部品をビルドする)
import os
os.chdir('/content')
!git clone -q -b main --recursive https://github.com/microsoft/TRELLIS.2.git
%cd /content/TRELLIS.2
!bash -c ". ./setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm" > /content/trellis_setup.log 2>&1; tail -5 /content/trellis_setup.log
# 入れ替わったライブラリを読み込み直すため、セッションを再起動する(「クラッシュ」と出るが正常)。次のセルから続ける
os.kill(os.getpid(), 9)

In [ ]:
# TRELLIS.2: 記録の道具(再起動したので、もう一度読み込む)
import os, json, time, traceback, gc, sys, importlib.util
# Colab(Python 3.13・torch 2.11)では setup.sh の flash-attn 2.7.3 を組み立てられない。flash-attn が無ければ、
# ふつうの注意は PyTorch の sdpa、疎な注意は xformers にする(TRELLIS.2 を読み込む前に決める)
if importlib.util.find_spec('flash_attn') is None:
    !pip install -q xformers --no-deps
    os.environ['ATTN_BACKEND'] = 'sdpa'
    os.environ['SPARSE_ATTN_BACKEND'] = 'xformers'
import torch
print('torch', torch.__version__, 'attn', os.environ.get('ATTN_BACKEND', 'flash_attn'))
# TRELLIS.2 は申請制(gated)の DINOv3 を使う。Hugging Face で利用を申請し、Colab のシークレット HF_TOKEN に
# トークン(Read)を入れておく(トークンをノートブックに直接書かない)
from google.colab import userdata
from huggingface_hub import login
try:
    login(userdata.get('HF_TOKEN'))
except Exception as exc:
    print('HF_TOKEN でログインできません(DINOv3 を読めず失敗します):', exc)
if not os.path.isdir('/content/inputs'):
    # ランタイムを変えた(A100 など)ときは、入力をもう一度アップロードする
    import zipfile
    from google.colab import files
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('/content')
os.makedirs('/content/results', exist_ok=True)
LOG = '/content/results/log.json'
def log(char, method, **kw):
    data = json.load(open(LOG)) if os.path.exists(LOG) else []
    data.append({'char': char, 'method': method, 'gpu': torch.cuda.get_device_name(0), **kw})
    json.dump(data, open(LOG, 'w'), ensure_ascii=False, indent=1)
    print(char, method, kw)
def run(char, method, fn):
    torch.cuda.reset_peak_memory_stats(); t = time.time()
    try:
        fn(); log(char, method, ok=True, seconds=round(time.time() - t, 1),
                  vram_gb=round(torch.cuda.max_memory_allocated() / 2**30, 1))
    except Exception as exc:
        log(char, method, ok=False, error=f'{type(exc).__name__}: {exc}', trace=traceback.format_exc()[-1500:])
    gc.collect(); torch.cuda.empty_cache()
CHARS = sorted(os.listdir('/content/inputs'))

In [ ]:
# TRELLIS.2: 1枚(A ポーズ)から、テクスチャ付きの GLB
sys.path.insert(0, '/content/TRELLIS.2')
os.chdir('/content/TRELLIS.2')
from PIL import Image
try:
    from trellis2.pipelines import Trellis2ImageTo3DPipeline, rembg as trellis_rembg
    import o_voxel
    # 背景消しは申請の要る briaai/RMBG-2.0 が指定されているので、同じ仕組みで申請の要らない BiRefNet(MIT)にする
    # 元の __init__ はクラスに1回だけ残す(セルを何度実行しても、差し替えが自分を呼ばないように)
    if not hasattr(trellis_rembg.BiRefNet, '_original_init'):
        trellis_rembg.BiRefNet._original_init = trellis_rembg.BiRefNet.__init__
    def _open_birefnet(self, *args, **kwargs):
        trellis_rembg.BiRefNet._original_init(self,*['ZhengPeng7/BiRefNet' if 'RMBG' in str(a) else a for a in args],
                       **{k: 'ZhengPeng7/BiRefNet' if 'RMBG' in str(v) else v for k, v in kwargs.items()})
    trellis_rembg.BiRefNet.__init__ = _open_birefnet
    pipe = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B'); pipe.cuda()
    # BiRefNet の重みは fp16 で読まれるが、入力は float32 なので、そろえる
    pipe.rembg_model.model.float()
    # TRELLIS.2 は DINOv3 の embeddings・rope_embeddings・layer を直接使う。transformers の版で場所が違えば、古い名前で引けるようにする
    import transformers
    dino = pipe.image_cond_model.model
    print('transformers', transformers.__version__, 'DINOv3 の部品:', [n for n, _ in dino.named_children()])
    for want in ('embeddings', 'rope_embeddings', 'layer'):
        if not hasattr(dino, want):
            found = [(n, m) for n, m in dino.named_modules() if n.split('.')[-1] in (want, want + 's')]
            print(want, '->', [n for n, _ in found])
            if found:
                dino.__dict__[want] = found[0][1]
except Exception as exc:
    log('all', 'trellis2', ok=False, error=f'load: {type(exc).__name__}: {exc}', trace=traceback.format_exc()[-1500:])
    pipe = None
for char in CHARS if pipe else []:
    out = f'/content/results/{char}'; os.makedirs(out, exist_ok=True)
    def one():
        mesh = pipe.run(Image.open(f'/content/inputs/{char}/a_pose.png'))[0]
        mesh.simplify(16777216)
        glb = o_voxel.postprocess.to_glb(
            vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs, coords=mesh.coords,
            attr_layout=mesh.layout, voxel_size=mesh.voxel_size, aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
            decimation_target=300000, texture_size=2048, remesh=True, remesh_band=1, remesh_project=0, verbose=False)
        glb.export(f'{out}/trellis2.glb', extension_webp=True)
    run(char, 'trellis2', one)
print(open(LOG).read())

## 結果をまとめてダウンロード

In [ ]:
# 結果の zip(GLB・記録)。ビルドのログも入れる
import shutil, glob
for f in glob.glob('/content/*.log'):
    shutil.copy(f, '/content/results/')
shutil.make_archive('/content/m3d_results', 'zip', '/content', 'results')
from google.colab import files
files.download('/content/m3d_results.zip')